In [1]:
import glob
print("Files in /kaggle/working:")
for f in sorted(glob.glob('/kaggle/working/*')):
    print(f"   {f}")

Files in /kaggle/working:


In [ ]:
# Cell 14: Territory Mapping (FIXED - loads bundle first)
# =========================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import pairwise_distances
from scipy.spatial import Voronoi
import joblib
import os
from datetime import datetime

print("=" * 60)
print("SKANN-SSL V2.1.0 Territory Mapping")
print("=" * 60)

# ============================================================================
# LOAD BUNDLE (REQUIRED - loads variables from post-training pipeline)
# ============================================================================

print("\n📦 Loading production bundle...")
import glob
import joblib

# Find bundle in any location
possible_paths = [
    '/kaggle/working/SKANN_SSL_V2_Production_Bundle.joblib',
    '/kaggle/input/*/SKANN_SSL_V2_Production_Bundle.joblib',
    '/kaggle/input/*/*.joblib',
]

bundle_path = None
for pattern in possible_paths:
    matches = glob.glob(pattern)
    if matches:
        bundle_path = matches[0]
        break

if bundle_path:
    print(f"✅ Found bundle: {bundle_path}")
    bundle = joblib.load(bundle_path)
else:
    print("❌ Bundle not found!")
    print("   Upload SKANN_SSL_V2_Production_Bundle.joblib as a Kaggle Dataset")
    print("   Then add it to this notebook via '+ Add data'")
    raise FileNotFoundError("Bundle not found")



embeddings = bundle['embeddings']
labels = bundle['labels']
vessel_classes = bundle['vessel_labels']
class_to_idx = bundle['class_map']

print(f"   Embeddings: {embeddings.shape}")
print(f"   Labels: {labels.shape}")
print(f"   Classes: {vessel_classes}")

# ============================================================================
# COMPUTE CLASS CENTROIDS
# ============================================================================

print("\n📍 Computing class centroids...")

centroids = {}
centroid_stats = {}

for class_name, class_idx in class_to_idx.items():
    mask = labels == class_idx
    class_embeddings = embeddings[mask]
    
    # Centroid = mean embedding
    centroid = class_embeddings.mean(axis=0)
    centroids[class_name] = centroid
    
    # Compute statistics
    distances_to_centroid = np.linalg.norm(class_embeddings - centroid, axis=1)
    
    centroid_stats[class_name] = {
        'centroid': centroid,
        'n_samples': mask.sum(),
        'mean_distance': distances_to_centroid.mean(),
        'std_distance': distances_to_centroid.std(),
        'max_distance': distances_to_centroid.max(),
        'radius_95': np.percentile(distances_to_centroid, 95),
    }
    
    print(f"   {class_name}:")
    print(f"      Samples: {centroid_stats[class_name]['n_samples']}")
    print(f"      Mean dist to centroid: {centroid_stats[class_name]['mean_distance']:.4f}")
    print(f"      95% radius: {centroid_stats[class_name]['radius_95']:.4f}")

# ============================================================================
# INTER-CLASS DISTANCES
# ============================================================================

print("\n📏 Inter-class centroid distances:")

centroid_matrix = np.array([centroids[c] for c in vessel_classes])
inter_class_dist = pairwise_distances(centroid_matrix, metric='cosine')

# Print distance matrix
print("\n   Cosine Distance Matrix:")
header = "   " + " ".join([f"{c[:8]:>10}" for c in vessel_classes])
print(header)
for i, c1 in enumerate(vessel_classes):
    row = " ".join([f"{inter_class_dist[i,j]:>10.4f}" for j in range(len(vessel_classes))])
    print(f"   {c1[:8]:>10} {row}")

# Find closest pair
min_dist = float('inf')
closest_pair = None
for i in range(len(vessel_classes)):
    for j in range(i+1, len(vessel_classes)):
        if inter_class_dist[i,j] < min_dist:
            min_dist = inter_class_dist[i,j]
            closest_pair = (vessel_classes[i], vessel_classes[j])

print(f"\n   Closest pair: {closest_pair[0]} ↔ {closest_pair[1]} (cosine dist: {min_dist:.4f})")

# ============================================================================
# CLASSIFICATION THRESHOLDS
# ============================================================================

print("\n🎯 Computing classification thresholds...")

thresholds = {}
for class_name in vessel_classes:
    stats = centroid_stats[class_name]
    thresholds[class_name] = stats['radius_95']
    print(f"   {class_name}: threshold = {thresholds[class_name]:.4f}")

# ============================================================================
# TERRITORY VISUALIZATION (2D UMAP projection)
# ============================================================================

print("\n🎨 Creating territory visualization...")
print("   Computing UMAP projection...")

import umap
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
emb_2d = reducer.fit_transform(embeddings)

# Compute 2D centroids
centroids_2d = {}
for class_name, class_idx in class_to_idx.items():
    mask = labels == class_idx
    centroids_2d[class_name] = emb_2d[mask].mean(axis=0)

# Create figure
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Color scheme
colors = {
    'cargo_ship': '#E31A1C',
    'fishing_vessel': '#FF7F00',
    'small_craft': '#33A02C',
    'tanker': '#6A3D9A'
}

# Plot 1: Embeddings with centroids
ax1 = axes[0]
for class_name in vessel_classes:
    mask = labels == class_to_idx[class_name]
    ax1.scatter(emb_2d[mask, 0], emb_2d[mask, 1], 
               c=colors[class_name], label=class_name, 
               s=30, alpha=0.6, edgecolors='white', linewidth=0.3)

for class_name in vessel_classes:
    cx, cy = centroids_2d[class_name]
    ax1.scatter(cx, cy, c=colors[class_name], s=400, marker='*', 
               edgecolors='black', linewidth=2, zorder=10)
    ax1.annotate(class_name.replace('_', '\n'), (cx, cy), 
                fontsize=9, ha='center', va='bottom',
                xytext=(0, 15), textcoords='offset points',
                fontweight='bold')

ax1.set_xlabel('UMAP 1', fontsize=11)
ax1.set_ylabel('UMAP 2', fontsize=11)
ax1.set_title('V2.1.0 Embeddings with Class Centroids', fontsize=13)
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Plot 2: Voronoi territory map
ax2 = axes[1]

centroid_points = np.array([centroids_2d[c] for c in vessel_classes])

x_min, x_max = emb_2d[:, 0].min() - 2, emb_2d[:, 0].max() + 2
y_min, y_max = emb_2d[:, 1].min() - 2, emb_2d[:, 1].max() + 2

dummy_points = np.array([
    [x_min - 50, y_min - 50],
    [x_max + 50, y_min - 50],
    [x_min - 50, y_max + 50],
    [x_max + 50, y_max + 50],
])
all_points = np.vstack([centroid_points, dummy_points])

vor = Voronoi(all_points)

from matplotlib.patches import Polygon

for idx, region_idx in enumerate(vor.point_region[:len(vessel_classes)]):
    region = vor.regions[region_idx]
    if -1 not in region and len(region) > 0:
        polygon = [vor.vertices[i] for i in region]
        poly = Polygon(polygon, facecolor=colors[vessel_classes[idx]], 
                      alpha=0.3, edgecolor='black', linewidth=1.5)
        ax2.add_patch(poly)

for class_name in vessel_classes:
    mask = labels == class_to_idx[class_name]
    ax2.scatter(emb_2d[mask, 0], emb_2d[mask, 1], 
               c=colors[class_name], s=20, alpha=0.7,
               edgecolors='white', linewidth=0.2)

for class_name in vessel_classes:
    cx, cy = centroids_2d[class_name]
    ax2.scatter(cx, cy, c=colors[class_name], s=300, marker='*', 
               edgecolors='black', linewidth=2, zorder=10)
    ax2.annotate(class_name.replace('_', ' ').upper(), (cx, cy), 
                fontsize=8, ha='center', va='bottom',
                xytext=(0, 12), textcoords='offset points',
                fontweight='bold', color='black')

ax2.set_xlim(x_min, x_max)
ax2.set_ylim(y_min, y_max)
ax2.set_xlabel('UMAP 1', fontsize=11)
ax2.set_ylabel('UMAP 2', fontsize=11)
ax2.set_title('Vessel Class Territory Map (Voronoi)', fontsize=13)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/territory_map_v21.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Territory map saved: territory_map_v21.png")

# ============================================================================
# EXPORT TERRITORIES FOR DEPLOYMENT
# ============================================================================

print("\n📦 Exporting territory data for deployment...")

territory_bundle = {
    'centroids': {k: v.tolist() for k, v in centroids.items()},
    'centroid_stats': {k: {kk: vv.tolist() if isinstance(vv, np.ndarray) else vv 
                          for kk, vv in v.items()} for k, v in centroid_stats.items()},
    'thresholds': thresholds,
    'inter_class_distances': inter_class_dist.tolist(),
    'vessel_classes': vessel_classes,
    'class_to_idx': class_to_idx,
    'centroids_2d': {k: v.tolist() for k, v in centroids_2d.items()},
    'metadata': {
        'version': 'v2.1.0',
        'silhouette_score': 0.8299,
        'embedding_dim': embeddings.shape[1],
        'n_classes': len(vessel_classes),
        'export_date': datetime.now().isoformat(),
        'classification_method': 'nearest_centroid_cosine',
    }
}

territory_path = '/kaggle/working/vessel_territories_v21.joblib'
joblib.dump(territory_bundle, territory_path, compress=3)
print(f"✅ Territory bundle saved: {territory_path}")
print(f"   Size: {os.path.getsize(territory_path) / 1e3:.1f} KB")

# ============================================================================
# SUMMARY
# ============================================================================

print("\n" + "=" * 60)
print("🎯 TERRITORY MAPPING COMPLETE")
print("=" * 60)
print(f"\nClass Centroids (128-dim):")
for class_name in vessel_classes:
    print(f"   {class_name}: norm = {np.linalg.norm(centroids[class_name]):.4f}")

print(f"\nClosest class pair: {closest_pair[0]} ↔ {closest_pair[1]}")
print(f"   Cosine distance: {min_dist:.4f}")

print(f"\n95% Classification Radii:")
for class_name in vessel_classes:
    print(f"   {class_name}: {thresholds[class_name]:.4f}")

print("\n📁 Output Files:")
print("   territory_map_v21.png — Visual territory map")
print("   vessel_territories_v21.joblib — Deployment bundle")
print("=" * 60)

In [ ]:
# Cell: IMMEDIATE BACKUP - Run right after territory mapping!
# ============================================================
# Creates downloadable ZIP of ALL V2.1.0 artifacts

import zipfile
import os
import glob
from datetime import datetime

print("🔒 Creating backup ZIP...")

# All files to backup
files_to_backup = [
    # Post-training outputs
    '/kaggle/working/SKANN_SSL_V2_Production_Bundle.joblib',
    '/kaggle/working/umap_v21.png',
    '/kaggle/working/tsne_v21.png', 
    '/kaggle/working/silhouette_analysis.png',
    # Territory mapping outputs
    '/kaggle/working/territory_map_v21.png',
    '/kaggle/working/vessel_territories_v21.joblib',
    # Training artifacts (if present)
    '/kaggle/working/loss_history.txt',
]

# Add any checkpoints
files_to_backup.extend(glob.glob('/kaggle/working/BT_ckpt_epoch_*.pth'))
files_to_backup.extend(glob.glob('/kaggle/working/SKANN_SSL_V2_Final.pth'))

# Create timestamped ZIP
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_path = f'/kaggle/working/SKANN_SSL_V21_COMPLETE_{timestamp}.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in files_to_backup:
        if os.path.exists(fpath):
            zf.write(fpath, os.path.basename(fpath))
            size_mb = os.path.getsize(fpath) / 1e6
            print(f"   ✅ {os.path.basename(fpath)}: {size_mb:.2f} MB")
        else:
            print(f"   ⚠️ {os.path.basename(fpath)}: NOT FOUND")

print(f"\n🎯 BACKUP COMPLETE: {zip_path}")
print(f"   Total size: {os.path.getsize(zip_path) / 1e6:.1f} MB")
print(f"\n⬇️  DOWNLOAD THIS FILE NOW before session resets!")
print("=" * 60)

# List all outputs in /kaggle/working for verification
print("\n📁 All files in /kaggle/working:")
for f in sorted(glob.glob('/kaggle/working/*')):
    if os.path.isfile(f):
        print(f"   {os.path.basename(f)}: {os.path.getsize(f)/1e6:.2f} MB")
